<h1> Ridge Regression

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score, mean_squared_error

source_file_path = 'data/co2_source.xlsx'
source_sheets = ['Coal', 'Natural gas', 'Petroleum']

sector_file_path = 'data/co2_sector.xlsx'
sector_sheets = ['Residential', 'Commercial', 'Industrial', 'Transportation', 'Electric power', 'Total']

def load_excel_data(path, sheet_list):
    merged_df = None
    for sheet in sheet_list:
        
        df = pd.read_excel(path, sheet_name=sheet, skiprows=2)
        
        
        df_melt = df.melt(id_vars=['State'], var_name='Year', value_name=sheet)
        df_melt['Year'] = pd.to_numeric(df_melt['Year'], errors='coerce')
        
        if merged_df is None:
            merged_df = df_melt
        else:
            merged_df = pd.merge(merged_df, df_melt, on=['State', 'Year'], how='outer')
            
    
    merged_df = merged_df.dropna(subset=['Year'])
    merged_df['Year'] = merged_df['Year'].astype(int)
    merged_df = merged_df[merged_df['State'] != 'US'].dropna()
    return merged_df

df_sector = load_excel_data(sector_file_path, sector_sheets)
df_source = load_excel_data(source_file_path, source_sheets)

df_clean = pd.merge(df_sector, df_source, on=['State', 'Year'], how='inner')

X = df_clean[['Year', 'State']]
y = df_clean['Total']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['Year']),
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['State'])
    ])

ridge_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RidgeCV(alphas=np.logspace(-3, 3, 20)))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
ridge_pipeline.fit(X_train, y_train)

y_pred = ridge_pipeline.predict(X_test)
print(f"Model R-squared: {r2_score(y_test, y_pred):.4f}")
print(f"Best Alpha: {ridge_pipeline.named_steps['regressor'].alpha_:.4f}")

#Test Prediction
test_case = pd.DataFrame({'Year': [2025], 'State': ['TX']})
pred_val = ridge_pipeline.predict(test_case)
print(f"\nPredicted 2025 Total CO2 for TX: {pred_val[0]:.2f} Million Metric Tons")

Model R-squared: 0.9204
Best Alpha: 0.0785

Predicted 2025 Total CO2 for TX: 564.14 Million Metric Tons


<h3> Multiple Output Ridge Regression
<h4> Predicts Total Using Sum of All Sectors

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.multioutput import MultiOutputRegressor

source_file_path = 'data/co2_source.xlsx'
source_sheets = ['Coal', 'Natural gas', 'Petroleum']

sector_file_path = 'data/co2_sector.xlsx'
sector_sheets = ['Residential', 'Commercial', 'Industrial', 'Transportation', 'Electric power', 'Total']

def load_excel_data(path, sheet_list):
    merged_df = None
    for sheet in sheet_list:
        
        df = pd.read_excel(path, sheet_name=sheet, skiprows=2)
        
        
        df_melt = df.melt(id_vars=['State'], var_name='Year', value_name=sheet)
        df_melt['Year'] = pd.to_numeric(df_melt['Year'], errors='coerce')
        
        if merged_df is None:
            merged_df = df_melt
        else:
            merged_df = pd.merge(merged_df, df_melt, on=['State', 'Year'], how='outer')
            
    
    merged_df = merged_df.dropna(subset=['Year'])
    merged_df['Year'] = merged_df['Year'].astype(int)
    merged_df = merged_df[merged_df['State'] != 'US'].dropna()
    return merged_df

df_sector = load_excel_data(sector_file_path, sector_sheets)
df_source = load_excel_data(source_file_path, source_sheets)

df_clean = pd.merge(df_sector, df_source, on=['State', 'Year'], how='inner')

X = df_clean[['Year', 'State']]
y = df_clean[['Coal', 'Natural gas', 'Petroleum']]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['Year']),
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['State'])
    ])

ridge_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', MultiOutputRegressor(RidgeCV(alphas=np.logspace(-3, 3, 20))))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
ridge_pipeline.fit(X_train, y_train)

y_pred = ridge_pipeline.predict(X_test)

for i, target in enumerate(y.columns):
    print(f"Model R-squared for {target}: {r2_score(y_test[target], y_pred[:, i]):.4f}")

print(f"Best Alpha: {ridge_pipeline.named_steps['regressor'].estimators_[0].alpha_:.4f}")

#Test Prediction
test_case = pd.DataFrame({'Year': [2025], 'State': ['TX']})
pred_val = ridge_pipeline.predict(test_case)

print("\nPredicted 2025 Outputs for TX:")
print(f"Total CO2: {pred_val.sum(axis=1)[0]:.2f} Million Metric Tons ")
for i, target in enumerate(y.columns):
    sector_total_sum += pred_val[0, i]
    print(f"{target}: {pred_val[0, i]:.2f} Million Metric Tons")

Model R-squared for Coal: 0.8135
Model R-squared for Natural gas: 0.9509
Model R-squared for Petroleum: 0.9050
Best Alpha: 0.1624

Predicted 2025 Outputs for TX:
Total CO2: 564.18 Million Metric Tons 


NameError: name 'sector_total_sum' is not defined

<h3> Ridge Regression per State
<h4> Predicts total by only training on a single state's data

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score, mean_squared_error

source_file_path = 'data/co2_source.xlsx'
source_sheets = ['Coal', 'Natural gas', 'Petroleum']

sector_file_path = 'data/co2_sector.xlsx'
sector_sheets = ['Residential','Commercial','Industrial','Transportation','Electric power','Total']

other_file_path = 'data/indicator_other.xlsx'
other_sheets = ['Total population','Real GDP','HDD','CDD']

def load_excel_data(path, sheet_list):
    merged_df = None
    for sheet in sheet_list:
        
        df = pd.read_excel(path, sheet_name=sheet, skiprows=2)
        
        
        df_melt = df.melt(id_vars=['State'], var_name='Year', value_name=sheet)
        df_melt['Year'] = pd.to_numeric(df_melt['Year'], errors='coerce')
        
        if merged_df is None:
            merged_df = df_melt
        else:
            merged_df = pd.merge(merged_df, df_melt, on=['State', 'Year'], how='outer')
            
    
    merged_df = merged_df.dropna(subset=['Year'])
    merged_df['Year'] = merged_df['Year'].astype(int)
    merged_df = merged_df[merged_df['State'] != 'US'].dropna()
    return merged_df

df_sector = load_excel_data(sector_file_path, sector_sheets)
df_source = load_excel_data(source_file_path, source_sheets)
df_other = load_excel_data(other_file_path,other_sheets)

df_clean = pd.merge(df_sector, df_source, on=['State', 'Year'], how='inner')
df_clean = pd.merge(df_clean, df_other, on=['State', 'Year'], how='inner')

states = df_clean['State'].unique()

predictions = {}

df_clean['State'] = df_clean['State'].astype('category')
df_clean
states = df_clean['State'].unique()

predictions = {}

for state in states:
    # Filter data for the current state
    df_state = df_clean[df_clean['State'] == state].copy()
    
    # Features and target
    X = df_state[['Year']]
    y = df_state['Total']
    
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), ['Year'])
        ])


    ridge_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', RidgeCV(alphas=np.logspace(-3, 3, 20)))
    ])
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    ridge_pipeline.fit(X_train, y_train)
    
    test_case = pd.DataFrame({'Year': [2025]})
    pred_val = ridge_pipeline.predict(test_case)
    predictions[state] = pred_val[0]

# Print predictions
for state, pred in predictions.items():
    print(f"Predicted 2025 Total CO2 for {state}: {pred:.2f} Million Metric Tons")

Predicted 2025 Total CO2 for AK: 36.10 Million Metric Tons
Predicted 2025 Total CO2 for AL: 101.07 Million Metric Tons
Predicted 2025 Total CO2 for AR: 62.86 Million Metric Tons
Predicted 2025 Total CO2 for AZ: 89.97 Million Metric Tons
Predicted 2025 Total CO2 for CA: 322.24 Million Metric Tons
Predicted 2025 Total CO2 for CO: 88.73 Million Metric Tons
Predicted 2025 Total CO2 for CT: 33.24 Million Metric Tons
Predicted 2025 Total CO2 for DC: 2.16 Million Metric Tons
Predicted 2025 Total CO2 for DE: 12.06 Million Metric Tons
Predicted 2025 Total CO2 for FL: 227.84 Million Metric Tons
Predicted 2025 Total CO2 for GA: 123.79 Million Metric Tons
Predicted 2025 Total CO2 for HI: 18.76 Million Metric Tons
Predicted 2025 Total CO2 for IA: 72.47 Million Metric Tons
Predicted 2025 Total CO2 for ID: 20.68 Million Metric Tons
Predicted 2025 Total CO2 for IL: 187.21 Million Metric Tons
Predicted 2025 Total CO2 for IN: 155.38 Million Metric Tons
Predicted 2025 Total CO2 for KS: 58.52 Million Metr